# 2. Loading stock data with SQL

Our stock prices are stored in a database. We'll use SQL to select the observations we need, then load the results into pandas for analysis. Many SQL operations will look familiar from the pandas examples.

Download the bCourses database into this lecture's `data/` folder before running the notebook. You can start from a fresh kernel without running the other notebooks first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Connecting to the database

A database contains tables, and a connection lets Python run queries against them. The code below checks that the file exists and opens a read-only connection.

Once the data is loaded, we still need to check ticker names, date order, and prices before calculating returns.

In [ ]:
database = lesson_dir / 'data/data.db'
if not database.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses and place it at {database}')
conn = sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)
display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn))

In [ ]:
preview = pd.read_sql_query('SELECT ts, ticker, close, volume FROM ohlc LIMIT 5', conn)
display(preview)
display(preview.dtypes)

Each row represents one ticker on one date, so `(ticker, ts)` should uniquely identify it. `ts` is the date column. `ohlc` contains the reference prices, and `ohlc_hw` contains the homework data.

`LIMIT 5` returns five rows without guaranteeing their order. We'll add `ORDER BY` when we need chronological results.

The [ingestion reference](reference/2_data_ingestion_and_sql.ipynb) also covers loading CSV files and nested JSON.

### A first look at the homework data

We'll load six months of data to check the columns and values. `.info()` shows types and non-null counts; `.describe()` summarizes the numeric columns. Look at the ticker strings too, since inconsistent names can split one stock into several groups.

The database uses `ts` for dates, while the earlier cleaning example used `date`. Rename the column if you reuse that function. Also check `adj_close` separately from raw `close`. The counts below describe this sample, not the full database.

In [ ]:
hw_preview = pd.read_sql_query("SELECT * FROM ohlc_hw WHERE ts BETWEEN ? AND ? ORDER BY ticker, ts",
    conn, params=('2024-01-01', '2024-06-30'), parse_dates=['ts'])
hw_preview.info()
display(hw_preview.describe())
print('Sample range:', hw_preview['ts'].min(), hw_preview['ts'].max())
print('Raw ticker labels:', hw_preview['ticker'].nunique())
print('Example labels:', hw_preview['ticker'].unique()[:3])
display(hw_preview.isna().sum())
# Reference keys provide expected observations for each asset, without inventing holiday rows.
reference_keys = pd.read_sql_query('SELECT ticker, ts FROM ohlc WHERE ts BETWEEN ? AND ? ORDER BY ticker, ts',
    conn, params=('2024-01-01', '2024-06-30'), parse_dates=['ts'])

## 2. Selecting rows and columns

To retrieve AAPL's closing prices for the first half of 2024, we need to choose the columns, filter the ticker and dates, and sort the result.

| SQL | Equivalent pandas operation |
|---|---|
| `SELECT ts, close` | Select columns |
| `FROM ohlc` | Choose the table |
| `WHERE ...` | Filter rows with a mask |
| `ORDER BY ts` | `sort_values('ts')` |

Check that the output contains only AAPL and falls within the requested dates.

In [ ]:
query = """
SELECT ts, ticker, close, volume
FROM ohlc
WHERE ticker = 'AAPL'
  AND ts >= '2024-01-01' AND ts <= '2024-06-30'
ORDER BY ts
"""
aapl = pd.read_sql_query(query, conn, parse_dates=['ts'])
display(aapl.head())
print('Rows:', len(aapl), '| First date:', aapl['ts'].min(), '| Last date:', aapl['ts'].max())

### Passing query parameters

The `?` placeholders receive values from `params`. This lets us change the ticker or dates without constructing a new SQL string.

Try changing `selected_ticker` to MSFT and rerunning the cell. Restore AAPL afterwards.

In [ ]:
selected_ticker = 'AAPL'
start_date, end_date = '2024-01-01', '2024-06-30'
price_query = """
SELECT ts, ticker, close, volume FROM ohlc
WHERE ticker = ? AND ts BETWEEN ? AND ? ORDER BY ts
"""
selected = pd.read_sql_query(price_query, conn, params=(selected_ticker, start_date, end_date), parse_dates=['ts'])
selected.head()

## 3. Checking a SQL filter with pandas

We'll select AAPL rows where the close is above $180, then apply the same condition in pandas. `assert_frame_equal` checks that both approaches return the same rows, columns, values, and order.

In [ ]:
example_query = """
SELECT ts, ticker, close FROM ohlc
WHERE ticker = ? AND ts BETWEEN ? AND ? AND close > ? ORDER BY ts
"""
example_query_result = pd.read_sql_query(example_query, conn, params=('AAPL', start_date, end_date, 180), parse_dates=['ts'])
example_expected_filter = aapl.loc[aapl['close'] > 180, ['ts', 'ticker', 'close']].reset_index(drop=True)
pd.testing.assert_frame_equal(example_query_result, example_expected_filter)
display(example_query_result.head())

## 4. Grouping and joining tables

`GROUP BY ticker` returns one summary row per ticker, much like pandas' `groupby`. The next query summarizes AAPL, MSFT, and NVDA.

An average share price describes the prices in the sample. It does not tell us which stock had the best return.

In [ ]:
sql_summary = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS observations, AVG(close) AS mean_close, AVG(volume) AS mean_volume
FROM ohlc
WHERE ticker IN ('AAPL', 'MSFT', 'NVDA') AND ts BETWEEN ? AND ?
GROUP BY ticker ORDER BY ticker
""", conn, params=(start_date, end_date))
display(sql_summary)
assert np.isclose(sql_summary.set_index('ticker').loc['AAPL', 'mean_close'], aapl['close'].mean())

### Joining on ticker and date

The market-cap table has dated observations. Joining on ticker alone could match a price row with several market-cap dates and duplicate the price row. We'll join on both ticker and date.

A left join keeps every price row. When there is no market-cap value for that date, the result contains a null. How would an inner join change the result?

In [ ]:
joined = pd.read_sql_query("""
SELECT o.ts, o.ticker, o.close, m.market_capitalization / 1e9 AS market_cap_bn
FROM ohlc AS o
LEFT JOIN market_caps AS m ON o.ticker = m.ticker AND o.ts = m.ts
WHERE o.ticker = ? AND o.ts BETWEEN ? AND ?
ORDER BY o.ts
""", conn, params=('AAPL', start_date, end_date), parse_dates=['ts'])
print('Before:', len(aapl), '| After:', len(joined))
assert len(joined) == len(aapl), 'Investigate repeated keys in the lookup.'
display(joined.loc[joined['market_cap_bn'].notna()].head())
print('Dates with no matching cap:', joined['market_cap_bn'].isna().sum())

## 5. Checking an NVDA query result

We'll check the ticker, date range, chronological order, unique keys, and finite positive closes. These checks can find problems in the returned rows. To find missing dates, we would also need to compare the result with a list of expected sessions.

In [ ]:
example_extract = pd.read_sql_query(price_query, conn, params=('NVDA', start_date, end_date), parse_dates=['ts'])
assert not example_extract.empty and example_extract['ticker'].eq('NVDA').all()
assert example_extract['ts'].between(pd.Timestamp(start_date), pd.Timestamp(end_date)).all()
assert example_extract['ts'].is_monotonic_increasing
assert not example_extract.duplicated(['ticker', 'ts']).any()
assert np.isfinite(example_extract['close']).all() and example_extract['close'].gt(0).all()
display(example_extract.head())
print('NVDA rows passed the checks; calendar completeness and vendor accuracy still need separate evidence.')
# COUNT(column) excludes nulls; COUNT(*) includes every row.
example_missing_volume = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS rows, COUNT(*) - COUNT(volume) AS missing_volumes
FROM ohlc_hw GROUP BY ticker ORDER BY ticker
""", conn)
display(example_missing_volume.head())

### Checks for the homework

For HW2, you can use each ticker's `reference_keys` to check for missing dates. You may instead use an exchange calendar if you explain the ticker's active period and your calendar assumptions. Weekdays include exchange holidays, and a correct check may find no missing dates.

Investigate unusual returns and inconsistent OHLC rows before calculating risk. The [stock report](3_stock_analysis_template.ipynb) includes checks for raw OHLC and volume as well.

## 6. Saving a table to the database

The next cell writes a summary to a separate database and reads it back to check the result. It leaves the source database unchanged.

For HW2, save your validated data as `ohlc_hw_cleaned` in a local database copy. `if_exists='replace'` overwrites the named table, so check the table name carefully. Store dates consistently as `YYYY-MM-DD`, using `.dt.strftime('%Y-%m-%d')`, so the report's SQL date filters work as expected.

The [ingestion reference](reference/2_data_ingestion_and_sql.ipynb) has more examples of SQL queries and file formats.

In [ ]:
output_dir = lesson_dir / 'outputs'
output_dir.mkdir(exist_ok=True)
demo_conn = sqlite3.connect(output_dir / 'cleaning_demo.db')
sql_summary.to_sql('ticker_summary', demo_conn, if_exists='replace', index=False)
roundtrip = pd.read_sql_query('SELECT * FROM ticker_summary ORDER BY ticker', demo_conn)
pd.testing.assert_frame_equal(roundtrip, sql_summary)
demo_conn.close()

## Exercises

Replace the placeholders with your answers, then run the checks below each exercise.

### Exercise E: filter MSFT prices

Retrieve `ts`, `ticker`, and `close` for MSFT in the first half of 2024, keeping only rows where the close exceeds $400. Order the result by date.

Store the query in `your_query` and the result in `your_query_result`. Check it against the equivalent pandas selection. You can use `AND close > 400`, or pass the price threshold as another query parameter.

In [ ]:
your_query = None
your_query_result = None
# your_query = """SELECT ... FROM ... WHERE ... ORDER BY ..."""
# your_query_result = pd.read_sql_query(your_query, conn, parse_dates=['ts'])

In [ ]:
msft = pd.read_sql_query(price_query, conn, params=('MSFT', start_date, end_date), parse_dates=['ts'])
expected_filter = msft.loc[msft['close'] > 400, ['ts', 'ticker', 'close']].reset_index(drop=True)
if your_query_result is None:
    print('Try E, then compare the SQL result with this pandas selection.')
else:
    pd.testing.assert_frame_equal(your_query_result.reset_index(drop=True), expected_filter)
    print('E passed: SQL and pandas give the same rows, columns, values, and order.')
display(expected_filter.head())

### Exercise: load JPM prices

Use `price_query` to retrieve JPM observations for the first half of 2024 and store them in `your_extract`. Check the ticker, date order, duplicate `(ticker, ts)` keys, and missing or nonpositive closes.

In `your_extract_note`, describe your checks and anything they leave unanswered, such as whether all expected dates are present. Useful methods include `.is_monotonic_increasing`, `.duplicated(['ticker', 'ts'])`, and `.isna()`.

For extra practice, use `COUNT(*) - COUNT(close)` to count missing closes per ticker in `ohlc_hw`.

In [ ]:
your_extract = None
your_extract_note = ''

In [ ]:
if your_extract is None:
    print('Prepare the JPM extract and a short quality note.')
else:
    assert not your_extract.empty and your_extract['ticker'].eq('JPM').all()
    assert your_extract['ts'].between(pd.Timestamp(start_date), pd.Timestamp(end_date)).all()
    assert your_extract['ts'].is_monotonic_increasing
    assert not your_extract.duplicated(['ticker', 'ts']).any()
    assert np.isfinite(your_extract['close']).all() and your_extract['close'].gt(0).all()
    assert your_extract_note.strip()
    print('Extract checks passed. These checks do not prove complete trading-calendar coverage.')